In [ ]:
import os
import shutil
import zipfile
from pathlib import Path
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

In [ ]:
### TODO 1: Split the training data into folders train and test with subfolders for each class
### Additionally, also convert the data (normalise the pixel values)

# Unzip the data
zip_path = 'archive.zip'
extract_dir = 'homer_bart_1'
if not os.path.exists(extract_dir):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('.')

# Prepare train and test directories
base_dir = Path('data_homer_bart')
train_dir = base_dir / 'train'
test_dir = base_dir / 'test'

if not base_dir.exists():
    for d in [train_dir, test_dir]:
        (d / 'homer').mkdir(parents=True, exist_ok=True)
        (d / 'bart').mkdir(parents=True, exist_ok=True)
    
    # Get all images
    all_images = list(Path(extract_dir).glob('*.bmp'))
    random.shuffle(all_images)
    
    # Split 80/20
    split_idx = int(0.8 * len(all_images))
    train_imgs = all_images[:split_idx]
    test_imgs = all_images[split_idx:]
    
    # Move files
    for img_list, target_dir in [(train_imgs, train_dir), (test_imgs, test_dir)]:
        for img in img_list:
            class_name = 'homer' if 'homer' in img.name.lower() else 'bart'
            shutil.copy(img, target_dir / class_name / img.name)

# Define transforms
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

train_dataset = datasets.ImageFolder(train_dir, transform=transform)
test_dataset = datasets.ImageFolder(test_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
input_shape = (3, 64, 64) # Check dimensions of the images
flattened_size = 3 * 64 * 64

# TODO 2: Add more layers to the neural network (start with flattening it)
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        # YOUR CODE HERE
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(flattened_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 2) # Optionally you could also have a layer with only 1 node with activation function as sigmoid
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()
print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# TODO 3: Modify train_data and the value of epochs accordingly
epochs = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    print(f'Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}')

In [ ]:
model.eval()
correct = 0
total = 0
test_loss = 0.0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()
        
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_acc = correct / total
print(f'Test accuracy: {test_acc:.4f}')